## Find Company Table

In [9]:
import json
from pathlib import Path

PROCESSED = Path("../data/processed")


def load_doc(ticker):
	path = next(PROCESSED.glob(f"{ticker}_10-K_*.json"))
	return json.loads(path.read_text(encoding="utf-8"))


def find_tables(ticker, keyword, n=3, max_rows=15):
	"""Show the first n tables of a company that contain the keyword."""
	doc = load_doc(ticker)
	hits = [
		b for b in doc["blocks"]
		if b["type"] == "table" and keyword.lower() in b["content"].lower()
	]
	print(f"{ticker}: {len(hits)} tables containing {keyword!r}\n")
	for i, b in enumerate(hits[:n]):
		print(f"--- table {i} ---")
		print("CONTEXT:", b["context"][-200:])
		print("\n".join(b["content"].splitlines()[:max_rows]))
		print()


find_tables("JPM", "Net income", n=1)

JPM: 23 tables containing 'Net income'

--- table 0 ---
CONTEXT: Financial THREE-YEAR SUMMARY OF CONSOLIDATED FINANCIAL HIGHLIGHTS (unaudited) As of or for the year ended December 31, (in millions, except per share, ratio, employee data and where otherwise noted)
|  | 2025 Selected income statement data | 2024 Selected income statement data | 2023 Selected income statement data |
| --- | --- | --- | --- |
| Total net revenue | 182,447 | 177,556 | 158,104 |
| Total noninterest expense | 95,640 | 91,797 | 87,172 |
| Pre-provision profit (a) | 86,807 | 85,759 | 70,932 |
| Provision for credit losses | 14,212 | 10,678 | 9,320 |
| Income before income tax expense | 72,595 | 75,081 | 61,612 |
| Income tax expense | 15,547 | 16,610 | 12,060 |
| Net income | 57,048 | 58,471 | 49,552 |
| Earnings per share data |  |  |  |
| Net income:              Basic | 20.05 | 19.79 | 16.25 |
| Diluted | 20.02 | 19.75 | 16.23 |
| Average shares:     Basic | 2,776.5 | 2,873.9 | 2,938.6 |
| Diluted | 2,781.5 

## Evaluation Set Inspection

In [16]:
"""For each question, counts chunks from the same company that contain every evidence string.
Zero means the evidence is wrong."""
with open("../data/chunks.jsonl", encoding="utf-8") as f:
	chunks = [json.loads(line) for line in f]
with open("../eval/questions.jsonl", encoding="utf-8") as f:
	questions = [json.loads(line) for line in f if line.strip()]

problems = 0
for q in questions:
	if not q["evidence"]:
		continue
	matches = sum(
		c["ticker"] == q["ticker"]
		and all(e.lower() in c["text"].lower() for e in q["evidence"])
		for c in chunks
	)
	if matches == 0:
		problems += 1
		print("NO MATCH:", q["id"], q["evidence"])

## Expand a Table into Rows

In [11]:
def table_rows(ticker, keyword, table_i=0):
	"""Return the header and rows of one table as lists of cells."""
	doc = load_doc(ticker)
	hits = [
		b for b in doc["blocks"]
		if b["type"] == "table" and keyword.lower() in b["content"].lower()
	]
	lines = hits[table_i]["content"].splitlines()
	split = lambda line: [c.strip() for c in line.strip().strip("|").split("|")]
	header = split(lines[0])
	rows = [split(line) for line in lines[2:]]   # skip the "| --- |" separator

	print("CONTEXT:", hits[table_i]["context"][-150:])
	print("COLUMNS:", list(enumerate(header)))
	for i, r in enumerate(rows[:25]):
		print(i, r)
	return header, rows


## Build One Question Line

In [12]:
def make_question(qid, ticker, question, header, row, col, category="lookup"):
	"""Build one JSONL line, taking the answer and evidence straight from the table."""
	label, value = row[0], row[col]   # takes the label and value directly from the table
	q = {
		"id": qid,
		"ticker": ticker,
		"category": category,
		"question": question,
		"answer": value,
		"evidence": [label, value],
	}
	print(f"# check: row '{label}', column '{header[col]}' -> {value}")
	print(json.dumps(q, ensure_ascii=False))

In [13]:
find_tables("BAC", "Net interest income")

BAC: 17 tables containing 'Net interest income'

--- table 0 ---
CONTEXT: e accounting method changes. For more information, see Note 1 – Summary of Significant Accounting Principles to the Consolidated Financial Statements and Exhibit 18 to this Annual Report on Form 10-K.
| Table 1 Summary Income Statement and Selected Financial Data (Dollars in millions, except per share information) | 2025 | 2024 |
| --- | --- | --- |
| Income statement |  |  |
| Net interest income | 60,096 | 56,060 |
| Noninterest income | 53,001 | 49,796 |
| Total revenue, net of interest expense | 113,097 | 105,856 |
| Provision for credit losses | 5,675 | 5,821 |
| Noninterest expense | 69,727 | 66,812 |
| Income before income taxes | 37,695 | 33,223 |
| Income tax expense | 7,186 | 6,250 |
| Net income | 30,509 | 26,973 |
| Preferred stock dividends and other | 1,454 | 1,629 |
| Net income applicable to common shareholders | 29,055 | 25,344 |
| Per common share information |  |  |
| Earnings | 3.86 | 3.23 |



In [18]:
header, rows = table_rows("BAC", "Net interest income", table_i=1)

CONTEXT: on page 27, Table 6 on page 32 and Table 7 on page 33. For information on key segment performance metrics, see Business Segment Operations on page 36.
COLUMNS: [(0, 'Table 6 Selected Annual Financial Data (In millions, except per share information)'), (1, '2025'), (2, '2024'), (3, '2023')]
0 ['Income statement', '', '', '']
1 ['Net interest income', '60,096', '56,060', '56,931']
2 ['Noninterest income', '53,001', '49,796', '45,838']
3 ['Total revenue, net of interest expense', '113,097', '105,856', '102,769']
4 ['Provision for credit losses', '5,675', '5,821', '4,394']
5 ['Noninterest expense', '69,727', '66,812', '65,845']
6 ['Income before income taxes', '37,695', '33,223', '32,530']
7 ['Income tax expense', '7,186', '6,250', '6,225']
8 ['Net income', '30,509', '26,973', '26,305']
9 ['Net income applicable to common shareholders', '29,055', '25,344', '24,656']
10 ['Average common shares issued and outstanding', '7,521.9', '7,855.5', '8,028.6']
11 ['Average diluted common sha

In [20]:
header, rows = table_rows("GS", "Net interest income", table_i=1)

CONTEXT: ACHS GROUP, INC. AND SUBSIDIARIES The table below presents our FICC and Equities net revenues by line item in the consolidated statements of earnings.
COLUMNS: [(0, '$ in millions'), (1, 'FICC'), (2, 'Equities')]
0 ['Year Ended December 2025', '', '']
1 ['Market making', '7,240', '10,765']
2 ['Commissions and fees', '–', '5,029']
3 ['Other principal transactions', '966', '67']
4 ['Net interest income', '6,316', '674']
5 ['Total', '14,522', '16,535']
6 ['Year Ended December 2024', '', '']
7 ['Market making', '9,020', '9,370']
8 ['Commissions and fees', '–', '4,289']
9 ['Other principal transactions', '1,256', '69']
10 ['Net interest income', '3,066', '(296)']
11 ['Total', '13,342', '13,432']
12 ['Year Ended December 2023', '', '']
13 ['Market making', '10,632', '7,606']
14 ['Commissions and fees', '–', '3,736']
15 ['Other principal transactions', '697', '81']
16 ['Net interest income', '821', '126']
17 ['Total', '12,150', '11,549']


In [8]:
make_question("bac-01", "BAC", "What was Bank of America's net interest income in 2025?", header, rows[3], 1)

# check: row 'Noninterest income', column '' -> 53,001
{"id": "bac-01", "ticker": "BAC", "category": "lookup", "question": "What was Bank of America's net interest income in 2025?", "answer": "53,001", "evidence": ["Noninterest income", "53,001"]}


In [15]:
make_question("bac-01", "BAC", "What was Bank of America's net interest income in 2025?", header, rows[1], 1)
make_question("bac-02", "BAC", "What was Bank of America's provision for credit losses in 2024?", header, rows[4], 2)
make_question("bac-03", "BAC", "How much profit did Bank of America keep after all costs and taxes in 2023?", header, rows[8], 3, category="paraphrase")

# check: row 'Net interest income', column '2025' -> 60,096
{"id": "bac-01", "ticker": "BAC", "category": "lookup", "question": "What was Bank of America's net interest income in 2025?", "answer": "60,096", "evidence": ["Net interest income", "60,096"]}
# check: row 'Provision for credit losses', column '2024' -> 5,821
{"id": "bac-02", "ticker": "BAC", "category": "lookup", "question": "What was Bank of America's provision for credit losses in 2024?", "answer": "5,821", "evidence": ["Provision for credit losses", "5,821"]}
# check: row 'Net income', column '2023' -> 26,305
{"id": "bac-03", "ticker": "BAC", "category": "paraphrase", "question": "How much profit did Bank of America keep after all costs and taxes in 2023?", "answer": "26,305", "evidence": ["Net income", "26,305"]}


In [33]:
find_tables("GS", "Net revenues")

GS: 10 tables containing 'Net revenues'

--- table 0 ---
CONTEXT: Financial Overview The table below presents an overview of our financial results and selected financial ratios.
| Year Ended December $ in millions, except per share amounts | 2025 | 2024 | 2023 |
| --- | --- | --- | --- |
| Net revenues | 58,283 | 53,512 | 46,254 |
| Pre-tax earnings | 21,852 | 18,397 | 10,739 |
| Net earnings | 17,176 | 14,276 | 8,516 |
| Net earnings to common | 16,300 | 13,525 | 7,907 |
| Diluted EPS | 51.32 | 40.54 | 22.87 |
| ROE | 15.0% | 12.7% | 7.5% |
| ROTE | 16.0% | 13.5% | 8.1% |
| Net earnings to average assets | 1.0% | 0.9% | 0.5% |
| Return on shareholders’ equity | 13.9% | 12.0% | 7.3% |
| Average equity to average assets | 7.0% | 7.1% | 7.5% |
| Dividend payout ratio | 27.3% | 28.4% | 45.9% |

--- table 1 ---
CONTEXT: Net Revenues The table below presents our net revenues by line item.
| Year Ended December $ in millions | 2025 | 2024 | 2023 |
| --- | --- | --- | --- |
| Investment banki

In [28]:
header0, rows0 = table_rows("GS", "Net revenues", table_i=0)   # summary table
header1, rows1 = table_rows("GS", "Net revenues", table_i=1)   # net revenues by line item

CONTEXT: Financial Overview The table below presents an overview of our financial results and selected financial ratios.
COLUMNS: [(0, '$ in millions, except per share amounts'), (1, 'Year Ended December 2025'), (2, 'Year Ended December 2024'), (3, 'Year Ended December 2023')]
0 ['Net revenues', '58,283', '53,512', '46,254']
1 ['Pre-tax earnings', '21,852', '18,397', '10,739']
2 ['Net earnings', '17,176', '14,276', '8,516']
3 ['Net earnings to common', '16,300', '13,525', '7,907']
4 ['Diluted EPS', '51.32', '40.54', '22.87']
5 ['ROE', '15.0%', '12.7%', '7.5%']
6 ['ROTE', '16.0%', '13.5%', '8.1%']
7 ['Net earnings to average assets', '1.0%', '0.9%', '0.5%']
8 ['Return on shareholders’ equity', '13.9%', '12.0%', '7.3%']
9 ['Average equity to average assets', '7.0%', '7.1%', '7.5%']
10 ['Dividend payout ratio', '27.3%', '28.4%', '45.9%']
CONTEXT: Net Revenues The table below presents our net revenues by line item.
COLUMNS: [(0, '$ in millions'), (1, 'Year Ended December 2025'), (2, 'Year 

In [29]:
make_question("gs-01", "GS", "How much did Goldman Sachs earn from investment banking in 2024?", header1, rows1[0], 2)
make_question("gs-02", "GS", "What was Goldman Sachs's net interest income in 2023?", header1, rows1[8], 3)
make_question("gs-03", "GS", "How much money did Goldman Sachs bring in across all of its businesses in 2025?", header0, rows0[0], 1, category="paraphrase")

# check: row 'Investment banking', column 'Year Ended December 2024' -> 7,738
{"id": "gs-01", "ticker": "GS", "category": "lookup", "question": "How much did Goldman Sachs earn from investment banking in 2024?", "answer": "7,738", "evidence": ["Investment banking", "7,738"]}
# check: row 'Net interest income', column 'Year Ended December 2023' -> 6,351
{"id": "gs-02", "ticker": "GS", "category": "lookup", "question": "What was Goldman Sachs's net interest income in 2023?", "answer": "6,351", "evidence": ["Net interest income", "6,351"]}
# check: row 'Net revenues', column 'Year Ended December 2025' -> 58,283
{"id": "gs-03", "ticker": "GS", "category": "paraphrase", "question": "How much money did Goldman Sachs bring in across all of its businesses in 2025?", "answer": "58,283", "evidence": ["Net revenues", "58,283"]}


In [30]:
find_tables("XOM", "Net income")
find_tables("CVX", "Net income")

XOM: 16 tables containing 'Net income'

--- table 0 ---
CONTEXT: Table of Contents Financial Table of Contents FINANCIAL INFORMATION
| (millions of dollars, except where stated otherwise) | 2025 | 2024 | 2023 |
| --- | --- | --- | --- |
| Sales and other operating revenue | 323,905 | 339,247 | 334,697 |
| Net income (loss) attributable to ExxonMobil | 28,844 | 33,680 | 36,010 |
| Earnings (loss) per common share (dollars) | 6.70 | 7.84 | 8.89 |
| Earnings (loss) per common share – assuming dilution (dollars) | 6.70 | 7.84 | 8.89 |
| Earnings (loss) to average ExxonMobil share of equity (percent) | 11.0 | 14.4 | 18.0 |
| Working capital | 11,052 | 21,683 | 31,293 |
| Ratio of current assets to current liabilities (times) | 1.15 | 1.31 | 1.48 |
| Additions to property, plant, and equipment (1) | 31,476 | 109,332 | 29,038 |
| Property, plant, and equipment, less allowances | 299,373 | 294,318 | 214,940 |
| Total assets | 448,980 | 453,475 | 376,317 |
| Exploration expenses, including dry 

In [32]:
find_tables("MSFT", "Total revenue")

MSFT: 1 tables containing 'Total revenue'

--- table 0 ---
CONTEXT: ITEM 8. FINANCIAL STATEMENTS AND SUPPLEMENTARY DATA INCOME STATEMENTS
| (In millions, except per share amounts) Year Ended June 30, | 2026 | 2025 | 2024 |
| --- | --- | --- | --- |
| Revenue - Product | 64,696 | 63,946 | 64,773 |
| Revenue - Service and other | 267,143 | 217,778 | 180,349 |
| Total revenue | 331,839 | 281,724 | 245,122 |
| Cost of revenue - Product | 12,098 | 13,501 | 15,272 |
| Cost of revenue - Service and other | 94,276 | 74,330 | 58,842 |
| Total cost of revenue | 106,374 | 87,831 | 74,114 |
| Gross margin | 225,465 | 193,893 | 171,008 |
| Research and development | 35,562 | 32,488 | 29,510 |
| Sales and marketing | 26,710 | 25,654 | 24,456 |
| General and administrative | 7,956 | 7,223 | 7,609 |
| Operating income | 155,237 | 128,528 | 109,433 |
| Other income (expense), net | 10,697 | (4,901) | (1,646) |
| Income before income taxes | 165,934 | 123,627 | 107,787 |

